# Creación de gráficas de Bland Altman

## 1) Creación de gráficas de Bland Altman para todas las muestras sin segmentar por control de calidad y diferenciadas por zona mineralógica

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


# ============================================================
# CONFIGURACIÓN GENERAL
# ============================================================

archivo_excel = (
    r"C:\Users\Diego\Desktop\Ultimo Esfuerzo"
    r"\Copia de Base de Datos Cuprochlor Sin T Final.xlsx"
)

hoja = "BD elim y reducido (2)"

col_muestra = "id muestra"
col_zona = "zona min"

# Zonas mineralógicas
zonas = ["CUO", "SSE", "PRI"]

# Carpeta de salida
carpeta_salida = Path(
    "graficos_bland_altman_unificados_sin_segmentar"
)

carpeta_salida.mkdir(
    exist_ok=True
)


# ============================================================
# CONFIGURACIÓN DE VARIABLES
# ============================================================

# ------------------------------------------------------------
# ANÁLISIS DE SOLUCIÓN
# ------------------------------------------------------------

var1_solucion = "ext_cut_anlz_ph"
var2_solucion = "ext_cut_anlz_col"


# ------------------------------------------------------------
# ANÁLISIS DE RIPIOS
# ------------------------------------------------------------

var1_ripios = "ext_cut_rip_ph"
var2_ripios = "ext_cut_rip_selec_col"


# ============================================================
# CONFIGURACIÓN DE TEXTOS
# ============================================================
#
# AQUÍ SE PUEDE MODIFICAR:
#
# - Títulos
# - Nombre del eje X
# - Nombre del eje Y
# - Nombre de las leyendas
#
# ============================================================


# ------------------------------------------------------------
# ANÁLISIS DE SOLUCIÓN
# ------------------------------------------------------------

titulo_solucion = (
    "Método por cabeza analizada"
)

nombre_eje_x_solucion = (
    "Promedio de extracción [%]"
)

nombre_eje_y_solucion = (
    "Diferencia [%] "
    "(Ext Cut anlz PH - Ext Cut anlz Col)"
)

leyenda_media_solucion = "Media"

leyenda_ls_solucion = (
    "Límite superior"
)

leyenda_li_solucion = (
    "Límite inferior"
)

leyenda_cero_solucion = (
    "Diferencia = 0"
)


# ------------------------------------------------------------
# ANÁLISIS DE RIPIOS
# ------------------------------------------------------------

titulo_ripios = (
    "Método por tierras"
)

nombre_eje_x_ripios = (
    "Promedio de extracción [%]"
)

nombre_eje_y_ripios = (
    "Diferencia [%] "
    "(Ext Cut Rip PH - Ext Cut Rip Col)"
)

leyenda_media_ripios = "Media"

leyenda_ls_ripios = (
    "Límite superior"
)

leyenda_li_ripios = (
    "Límite inferior"
)

leyenda_cero_ripios = (
    "Diferencia = 0"
)


# ============================================================
# TÍTULO GENERAL DE CADA IMAGEN
# ============================================================

titulo_general = (
    "Gráficos de Bland-Altman - Zona {zona}"
)


# ============================================================
# CONFIGURACIÓN DEL TAMAÑO DE LAS LETRAS
# ============================================================

# Título de cada gráfico
tamano_titulo = 18

# Título general de la figura
tamano_titulo_general = 22

# Nombre de los ejes X
tamano_eje_x = 16

# Nombre de los ejes Y
tamano_eje_y = 16

# Números de los ejes
tamano_numeros_ejes = 14

# Tamaño de las leyendas
tamano_leyenda = 13


# ============================================================
# CONFIGURACIÓN DEL TAMAÑO DE LA FIGURA
# ============================================================

ancho_figura = 20
alto_figura = 8

# Resolución de salida
dpi_salida = 300


# ============================================================
# CONFIGURACIÓN DEL RANGO DEL EJE Y
# ============================================================

# Margen adicional sobre el rango calculado
margen_y = 0.10


# ------------------------------------------------------------
# RANGOS MANUALES OPCIONALES
# ------------------------------------------------------------
#
# Si se quiere definir manualmente el rango Y de una zona:
#
# "CUO": (-20, 20)
#
# Si dejas None, se calcula automáticamente.
#
# ------------------------------------------------------------

rangos_y_manuales = {
    "CUO": None,
    "SSE": None,
    "PRI": None
}


# ============================================================
# LECTURA DEL ARCHIVO EXCEL
# ============================================================

df = pd.read_excel(
    archivo_excel,
    sheet_name=hoja
)


# Limpiar nombres de columnas
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
)


print("Columnas encontradas:")
print(df.columns.tolist())


# ============================================================
# FUNCIÓN PARA CALCULAR BLAND-ALTMAN
# ============================================================

def calcular_bland_altman(
    datos,
    var1,
    var2
):

    datos = datos.copy()

    # --------------------------------------------------------
    # PROMEDIO ENTRE AMBOS MÉTODOS
    # --------------------------------------------------------

    datos["promedio"] = (
        datos[var1] +
        datos[var2]
    ) / 2


    # --------------------------------------------------------
    # DIFERENCIA ENTRE AMBOS MÉTODOS
    # --------------------------------------------------------

    datos["diferencia"] = (
        datos[var1] -
        datos[var2]
    )


    # --------------------------------------------------------
    # MEDIA DE LAS DIFERENCIAS
    # --------------------------------------------------------

    media_dif = (
        datos["diferencia"].mean()
    )


    # --------------------------------------------------------
    # DESVIACIÓN ESTÁNDAR
    # --------------------------------------------------------

    sd_dif = (
        datos["diferencia"].std()
    )


    # --------------------------------------------------------
    # LÍMITES DE CONCORDANCIA
    # --------------------------------------------------------

    limite_superior = (
        media_dif +
        1.96 * sd_dif
    )

    limite_inferior = (
        media_dif -
        1.96 * sd_dif
    )


    return (
        datos,
        media_dif,
        limite_superior,
        limite_inferior
    )


# ============================================================
# PROCESAMIENTO POR ZONA MINERALÓGICA
# ============================================================

for zona in zonas:

    print("\n" + "=" * 70)

    print(
        f"PROCESANDO ZONA: {zona}"
    )

    print("=" * 70)


    # ========================================================
    # FILTRAR DATOS DE LA ZONA
    # ========================================================

    datos_zona = df[
        df[col_zona]
        .astype(str)
        .str.strip()
        .str.upper()
        == zona
    ].copy()


    # ========================================================
    # DATOS PARA ANÁLISIS DE SOLUCIÓN
    # ========================================================

    datos_solucion = datos_zona[
        [
            col_muestra,
            col_zona,
            var1_solucion,
            var2_solucion
        ]
    ].dropna()


    # ========================================================
    # DATOS PARA ANÁLISIS DE RIPIOS
    # ========================================================

    datos_ripios = datos_zona[
        [
            col_muestra,
            col_zona,
            var1_ripios,
            var2_ripios
        ]
    ].dropna()


    # ========================================================
    # VERIFICAR DATOS SUFICIENTES
    # ========================================================

    if len(datos_solucion) < 2:

        print(
            f"{zona} - Solución: "
            "datos insuficientes"
        )

        continue


    if len(datos_ripios) < 2:

        print(
            f"{zona} - Ripios: "
            "datos insuficientes"
        )

        continue


    # ========================================================
    # CÁLCULO BLAND-ALTMAN: SOLUCIÓN
    # ========================================================

    (
        datos_solucion,
        media_solucion,
        ls_solucion,
        li_solucion
    ) = calcular_bland_altman(
        datos_solucion,
        var1_solucion,
        var2_solucion
    )


    # ========================================================
    # CÁLCULO BLAND-ALTMAN: RIPIOS
    # ========================================================

    (
        datos_ripios,
        media_ripios,
        ls_ripios,
        li_ripios
    ) = calcular_bland_altman(
        datos_ripios,
        var1_ripios,
        var2_ripios
    )


    # ========================================================
    # CALCULAR RANGO Y COMÚN
    # ========================================================
    #
    # Se consideran los valores de ambos análisis.
    #
    # Esto garantiza que los gráficos de una misma zona
    # tengan exactamente el mismo rango en el eje Y.
    #
    # ========================================================

    valores_y = [

        # ----------------------------------------------------
        # SOLUCIÓN
        # ----------------------------------------------------

        datos_solucion[
            "diferencia"
        ].min(),

        datos_solucion[
            "diferencia"
        ].max(),

        media_solucion,

        ls_solucion,

        li_solucion,


        # ----------------------------------------------------
        # RIPIOS
        # ----------------------------------------------------

        datos_ripios[
            "diferencia"
        ].min(),

        datos_ripios[
            "diferencia"
        ].max(),

        media_ripios,

        ls_ripios,

        li_ripios
    ]


    y_min = min(
        valores_y
    )

    y_max = max(
        valores_y
    )


    # Rango total
    rango_y = (
        y_max -
        y_min
    )


    # Agregar margen
    y_min = (
        y_min -
        margen_y * rango_y
    )

    y_max = (
        y_max +
        margen_y * rango_y
    )


    # ========================================================
    # APLICAR RANGO MANUAL SI EXISTE
    # ========================================================

    if (
        rangos_y_manuales
        .get(zona)
        is not None
    ):

        y_min, y_max = (
            rangos_y_manuales[zona]
        )


    print(
        f"\nRango Y utilizado para {zona}: "
        f"{y_min:.2f} a {y_max:.2f}"
    )


    # ========================================================
    # CREAR FIGURA CON DOS GRÁFICOS
    # ========================================================
    #
    # IZQUIERDA  = SOLUCIÓN
    # DERECHA    = RIPIOS
    #
    # ========================================================

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(
            ancho_figura,
            alto_figura
        ),
        sharey=True
    )


    # ========================================================
    # GRÁFICO IZQUIERDO
    # ANÁLISIS DE SOLUCIÓN
    # ========================================================

    ax1 = axes[0]


    # --------------------------------------------------------
    # PUNTOS
    # --------------------------------------------------------

    ax1.scatter(
        datos_solucion[
            "promedio"
        ],
        datos_solucion[
            "diferencia"
        ]
    )


    # --------------------------------------------------------
    # MEDIA
    # --------------------------------------------------------

    ax1.axhline(
        media_solucion,
        linestyle="--",
        label=(
            f"{leyenda_media_solucion} = "
            f"{media_solucion:.2f}"
        )
    )


    # --------------------------------------------------------
    # LÍMITE SUPERIOR
    # --------------------------------------------------------

    ax1.axhline(
        ls_solucion,
        linestyle="--",
        label=(
            f"{leyenda_ls_solucion} = "
            f"{ls_solucion:.2f}"
        )
    )


    # --------------------------------------------------------
    # LÍMITE INFERIOR
    # --------------------------------------------------------

    ax1.axhline(
        li_solucion,
        linestyle="--",
        label=(
            f"{leyenda_li_solucion} = "
            f"{li_solucion:.2f}"
        )
    )


    # --------------------------------------------------------
    # LÍNEA CERO
    # --------------------------------------------------------

    ax1.axhline(
        0,
        linestyle="-",
        label=(
            leyenda_cero_solucion
        )
    )


    # --------------------------------------------------------
    # EJE X
    # --------------------------------------------------------

    ax1.set_xlabel(
        nombre_eje_x_solucion,
        fontsize=tamano_eje_x
    )


    # --------------------------------------------------------
    # EJE Y
    # --------------------------------------------------------

    ax1.set_ylabel(
        nombre_eje_y_solucion,
        fontsize=tamano_eje_y
    )


    # --------------------------------------------------------
    # TÍTULO
    # --------------------------------------------------------

    ax1.set_title(
        titulo_solucion,
        fontsize=tamano_titulo
    )


    # --------------------------------------------------------
    # RANGO Y
    # --------------------------------------------------------

    ax1.set_ylim(
        y_min,
        y_max
    )


    # --------------------------------------------------------
    # NÚMEROS DE LOS EJES
    # --------------------------------------------------------

    ax1.tick_params(
        axis="both",
        labelsize=tamano_numeros_ejes
    )


    # --------------------------------------------------------
    # LEYENDA
    # --------------------------------------------------------

    ax1.legend(
        fontsize=tamano_leyenda
    )


    # ========================================================
    # GRÁFICO DERECHO
    # ANÁLISIS DE RIPIOS
    # ========================================================

    ax2 = axes[1]


    # --------------------------------------------------------
    # PUNTOS
    # --------------------------------------------------------

    ax2.scatter(
        datos_ripios[
            "promedio"
        ],
        datos_ripios[
            "diferencia"
        ]
    )


    # --------------------------------------------------------
    # MEDIA
    # --------------------------------------------------------

    ax2.axhline(
        media_ripios,
        linestyle="--",
        label=(
            f"{leyenda_media_ripios} = "
            f"{media_ripios:.2f}"
        )
    )


    # --------------------------------------------------------
    # LÍMITE SUPERIOR
    # --------------------------------------------------------

    ax2.axhline(
        ls_ripios,
        linestyle="--",
        label=(
            f"{leyenda_ls_ripios} = "
            f"{ls_ripios:.2f}"
        )
    )


    # --------------------------------------------------------
    # LÍMITE INFERIOR
    # --------------------------------------------------------

    ax2.axhline(
        li_ripios,
        linestyle="--",
        label=(
            f"{leyenda_li_ripios} = "
            f"{li_ripios:.2f}"
        )
    )


    # --------------------------------------------------------
    # LÍNEA CERO
    # --------------------------------------------------------

    ax2.axhline(
        0,
        linestyle="-",
        label=(
            leyenda_cero_ripios
        )
    )


    # --------------------------------------------------------
    # EJE X
    # --------------------------------------------------------

    ax2.set_xlabel(
        nombre_eje_x_ripios,
        fontsize=tamano_eje_x
    )


    # --------------------------------------------------------
    # EJE Y
    # --------------------------------------------------------

    ax2.set_ylabel(
        nombre_eje_y_ripios,
        fontsize=tamano_eje_y
    )


    # --------------------------------------------------------
    # TÍTULO
    # --------------------------------------------------------

    ax2.set_title(
        titulo_ripios,
        fontsize=tamano_titulo
    )


    # --------------------------------------------------------
    # MISMO RANGO Y
    # --------------------------------------------------------

    ax2.set_ylim(
        y_min,
        y_max
    )


    # --------------------------------------------------------
    # NÚMEROS DE LOS EJES
    # --------------------------------------------------------

    ax2.tick_params(
        axis="both",
        labelsize=tamano_numeros_ejes
    )


    # --------------------------------------------------------
    # LEYENDA
    # --------------------------------------------------------

    ax2.legend(
        fontsize=tamano_leyenda
    )


    # ========================================================
    # TÍTULO GENERAL
    # ========================================================

    fig.suptitle(
        titulo_general.format(
            zona=zona
        ),
        fontsize=tamano_titulo_general,
        y=1.02
    )


    # ========================================================
    # AJUSTAR ESPACIADO
    # ========================================================

    plt.tight_layout()


    # ========================================================
    # GUARDAR IMAGEN UNIFICADA
    # ========================================================

    archivo_png = (
        carpeta_salida /
        f"bland_altman_{zona}.png"
    )


    plt.savefig(
        archivo_png,
        dpi=dpi_salida,
        bbox_inches="tight"
    )


    plt.close()


    # ========================================================
    # MOSTRAR RESULTADO
    # ========================================================

    print(
        f"Guardado: {archivo_png}"
    )


    # ========================================================
    # RESUMEN DE RESULTADOS
    # ========================================================

    print(
        f"\n{zona} - ANÁLISIS DE SOLUCIÓN"
    )

    print(
        f"Media: {media_solucion:.2f}"
    )

    print(
        f"LS: {ls_solucion:.2f}"
    )

    print(
        f"LI: {li_solucion:.2f}"
    )


    print(
        f"\n{zona} - ANÁLISIS DE RIPIOS"
    )

    print(
        f"Media: {media_ripios:.2f}"
    )

    print(
        f"LS: {ls_ripios:.2f}"
    )

    print(
        f"LI: {li_ripios:.2f}"
    )


# ============================================================
# FINALIZACIÓN
# ============================================================

print("\n" + "=" * 70)

print(
    "PROCESO FINALIZADO"
)

print("=" * 70)

print(
    "Se generó una imagen por cada zona mineralógica."
)

print(
    "Cada imagen contiene los dos gráficos Bland-Altman:"
)

print(
    "IZQUIERDA: Análisis de solución"
)

print(
    "ext_cut_anlz_ph vs ext_cut_anlz_col"
)

print(
    "DERECHA: Análisis de ripios"
)

print(
    "ext_cut_rip_ph vs ext_cut_rip_selec_col"
)

print(
    "Ambos gráficos de cada zona utilizan "
    "el mismo rango Y."
)

print(
    "Los números del eje Y se mantienen visibles."
)

print(
    "Los tamaños de letra están configurados "
    "para facilitar su lectura en tesis y documentos."
)

print("=" * 70)

## 2) Creación de gráficas de Bland Altman para todas las muestras segmentado por control de calidad y diferenciadas por zona mineralógica

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# CONFIGURACIÓN GENERAL
# ============================================================

archivo_excel = r"C:\Users\Diego\calculo graficas 3.xlsx"
hoja = "BD Muestras sel"

# Columnas de identificación
col_muestra = "muestra"
col_zona = "zona min"

# Zonas mineralógicas
zonas = ["CUO", "SSE", "PRI"]

# Carpeta de salida
carpeta_salida = Path("graficos_bland_altman_unificados")
carpeta_salida.mkdir(exist_ok=True)


# ============================================================
# CONFIGURACIÓN DE VARIABLES
# ============================================================

# ANÁLISIS DE EXTRACCIÓN
var1_extraccion = "ext iso anlz"
var2_extraccion = "ext col anlz"

# ANÁLISIS DE RIPIOS
var1_ripios = "iso rip"
var2_ripios = "col rip"


# ============================================================
# CONFIGURACIÓN DE TEXTOS
# ============================================================

# ------------------------------------------------------------
# ANÁLISIS DE EXTRACCIÓN
# ------------------------------------------------------------

titulo_extraccion = "Método por cabeza analizada"

nombre_eje_x_extraccion = (
    "Promedio de extracción de cobre [%]"
)

nombre_eje_y_extraccion = (
    "Diferencia [%] (Ext Iso anlz - Ext Col anlz)"
)

leyenda_media_extraccion = "Media"
leyenda_ls_extraccion = "Límite superior"
leyenda_li_extraccion = "Límite inferior"
leyenda_cero_extraccion = "Diferencia = 0"


# ------------------------------------------------------------
# ANÁLISIS DE RIPIOS
# ------------------------------------------------------------

titulo_ripios = "Método por tierras"

nombre_eje_x_ripios = (
    "Promedio de extracción de cobre en ripio [%]"
)

nombre_eje_y_ripios = (
    "Diferencia [%] (Iso rip - Col rip)"
)

leyenda_media_ripios = "Media"
leyenda_ls_ripios = "Límite superior"
leyenda_li_ripios = "Límite inferior"
leyenda_cero_ripios = "Diferencia = 0"


# ------------------------------------------------------------
# TÍTULO GENERAL
# ------------------------------------------------------------

titulo_general = (
    "Gráficos de Bland-Altman - Zona {zona}"
)


# ============================================================
# CONFIGURACIÓN DEL TAMAÑO DE LAS LETRAS
# ============================================================

# Título de cada gráfico
tamano_titulo = 18

# Título general de la figura
tamano_titulo_general = 22

# Nombre de los ejes X
tamano_eje_x = 16

# Nombre de los ejes Y
tamano_eje_y = 16

# Números de los ejes
tamano_numeros_ejes = 14

# Tamaño de las leyendas
tamano_leyenda = 13


# ============================================================
# CONFIGURACIÓN DEL TAMAÑO DE LA FIGURA
# ============================================================

ancho_figura = 20
alto_figura = 8

# Resolución de salida
dpi_salida = 300


# ============================================================
# CONFIGURACIÓN DEL RANGO DEL EJE Y
# ============================================================

# Margen adicional sobre el rango calculado
margen_y = 0.10


# ------------------------------------------------------------
# RANGOS MANUALES OPCIONALES
# ------------------------------------------------------------
#
# Si quieres establecer manualmente el rango Y:
#
# "CUO": (-20, 20)
#
# Si dejas None, se calcula automáticamente.
#
# ------------------------------------------------------------

rangos_y_manuales = {
    "CUO": None,
    "SSE": None,
    "PRI": None
}


# ============================================================
# LECTURA DE DATOS
# ============================================================

df = pd.read_excel(
    archivo_excel,
    sheet_name=hoja
)

# Limpiar nombres de columnas
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
)

print("Columnas encontradas:")
print(df.columns.tolist())


# ============================================================
# FUNCIÓN PARA CALCULAR BLAND-ALTMAN
# ============================================================

def calcular_bland_altman(datos, var1, var2):

    datos = datos.copy()

    # Promedio entre ambos métodos
    datos["promedio"] = (
        datos[var1] + datos[var2]
    ) / 2

    # Diferencia entre ambos métodos
    datos["diferencia"] = (
        datos[var1] - datos[var2]
    )

    # Media de las diferencias
    media_dif = datos["diferencia"].mean()

    # Desviación estándar
    sd_dif = datos["diferencia"].std()

    # Límites de concordancia
    limite_superior = (
        media_dif + 1.96 * sd_dif
    )

    limite_inferior = (
        media_dif - 1.96 * sd_dif
    )

    return (
        datos,
        media_dif,
        limite_superior,
        limite_inferior
    )


# ============================================================
# PROCESAMIENTO POR ZONA MINERALÓGICA
# ============================================================

for zona in zonas:

    print("\n" + "=" * 60)
    print(f"PROCESANDO ZONA: {zona}")
    print("=" * 60)


    # ========================================================
    # FILTRAR DATOS DE LA ZONA
    # ========================================================

    datos_zona = df[
        df[col_zona]
        .astype(str)
        .str.strip()
        .str.upper()
        == zona
    ].copy()


    # ========================================================
    # DATOS DE ANÁLISIS DE EXTRACCIÓN
    # ========================================================

    datos_extraccion = datos_zona[
        [
            col_muestra,
            col_zona,
            var1_extraccion,
            var2_extraccion
        ]
    ].dropna()


    # ========================================================
    # DATOS DE ANÁLISIS DE RIPIOS
    # ========================================================

    datos_ripios = datos_zona[
        [
            col_muestra,
            col_zona,
            var1_ripios,
            var2_ripios
        ]
    ].dropna()


    # ========================================================
    # VERIFICAR DATOS SUFICIENTES
    # ========================================================

    if len(datos_extraccion) < 2:

        print(
            f"{zona} - Extracción: "
            "datos insuficientes"
        )

        continue


    if len(datos_ripios) < 2:

        print(
            f"{zona} - Ripios: "
            "datos insuficientes"
        )

        continue


    # ========================================================
    # CÁLCULO BLAND-ALTMAN: EXTRACCIÓN
    # ========================================================

    (
        datos_extraccion,
        media_ext,
        ls_ext,
        li_ext
    ) = calcular_bland_altman(
        datos_extraccion,
        var1_extraccion,
        var2_extraccion
    )


    # ========================================================
    # CÁLCULO BLAND-ALTMAN: RIPIOS
    # ========================================================

    (
        datos_ripios,
        media_rip,
        ls_rip,
        li_rip
    ) = calcular_bland_altman(
        datos_ripios,
        var1_ripios,
        var2_ripios
    )


    # ========================================================
    # CALCULAR RANGO Y COMÚN PARA AMBOS GRÁFICOS
    # ========================================================

    valores_y = [

        # Datos de extracción
        datos_extraccion["diferencia"].min(),
        datos_extraccion["diferencia"].max(),
        media_ext,
        ls_ext,
        li_ext,

        # Datos de ripios
        datos_ripios["diferencia"].min(),
        datos_ripios["diferencia"].max(),
        media_rip,
        ls_rip,
        li_rip
    ]

    y_min = min(valores_y)
    y_max = max(valores_y)

    # Rango total
    rango_y = y_max - y_min

    # Agregar margen
    y_min = (
        y_min -
        margen_y * rango_y
    )

    y_max = (
        y_max +
        margen_y * rango_y
    )


    # ========================================================
    # APLICAR RANGO MANUAL SI EXISTE
    # ========================================================

    if rangos_y_manuales.get(zona) is not None:

        y_min, y_max = (
            rangos_y_manuales[zona]
        )


    print(
        f"\nRango Y utilizado para {zona}: "
        f"{y_min:.2f} a {y_max:.2f}"
    )


    # ========================================================
    # CREAR FIGURA CON DOS GRÁFICOS
    # ========================================================

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(
            ancho_figura,
            alto_figura
        ),
        sharey=True
    )


    # ========================================================
    # GRÁFICO IZQUIERDO
    # ANÁLISIS DE EXTRACCIÓN
    # ========================================================

    ax1 = axes[0]

    # Puntos
    ax1.scatter(
        datos_extraccion["promedio"],
        datos_extraccion["diferencia"]
    )

    # Media
    ax1.axhline(
        media_ext,
        linestyle="--",
        label=(
            f"{leyenda_media_extraccion} = "
            f"{media_ext:.2f}"
        )
    )

    # Límite superior
    ax1.axhline(
        ls_ext,
        linestyle="--",
        label=(
            f"{leyenda_ls_extraccion} = "
            f"{ls_ext:.2f}"
        )
    )

    # Límite inferior
    ax1.axhline(
        li_ext,
        linestyle="--",
        label=(
            f"{leyenda_li_extraccion} = "
            f"{li_ext:.2f}"
        )
    )

    # Línea cero
    ax1.axhline(
        0,
        linestyle="-",
        label=leyenda_cero_extraccion
    )

    # Nombre eje X
    ax1.set_xlabel(
        nombre_eje_x_extraccion,
        fontsize=tamano_eje_x
    )

    # Nombre eje Y
    ax1.set_ylabel(
        nombre_eje_y_extraccion,
        fontsize=tamano_eje_y
    )

    # Título
    ax1.set_title(
        titulo_extraccion,
        fontsize=tamano_titulo
    )

    # Rango Y
    ax1.set_ylim(
        y_min,
        y_max
    )

    # Tamaño de números de ejes
    ax1.tick_params(
        axis="both",
        labelsize=tamano_numeros_ejes
    )

    # Leyenda
    ax1.legend(
        fontsize=tamano_leyenda
    )


    # ========================================================
    # GRÁFICO DERECHO
    # ANÁLISIS DE RIPIOS
    # ========================================================

    ax2 = axes[1]

    # Puntos
    ax2.scatter(
        datos_ripios["promedio"],
        datos_ripios["diferencia"]
    )

    # Media
    ax2.axhline(
        media_rip,
        linestyle="--",
        label=(
            f"{leyenda_media_ripios} = "
            f"{media_rip:.2f}"
        )
    )

    # Límite superior
    ax2.axhline(
        ls_rip,
        linestyle="--",
        label=(
            f"{leyenda_ls_ripios} = "
            f"{ls_rip:.2f}"
        )
    )

    # Límite inferior
    ax2.axhline(
        li_rip,
        linestyle="--",
        label=(
            f"{leyenda_li_ripios} = "
            f"{li_rip:.2f}"
        )
    )

    # Línea cero
    ax2.axhline(
        0,
        linestyle="-",
        label=leyenda_cero_ripios
    )

    # Nombre eje X
    ax2.set_xlabel(
        nombre_eje_x_ripios,
        fontsize=tamano_eje_x
    )

    # Nombre eje Y
    ax2.set_ylabel(
        nombre_eje_y_ripios,
        fontsize=tamano_eje_y
    )

    # Título
    ax2.set_title(
        titulo_ripios,
        fontsize=tamano_titulo
    )

    # Mismo rango Y
    ax2.set_ylim(
        y_min,
        y_max
    )

    # Tamaño de números de ejes
    ax2.tick_params(
        axis="both",
        labelsize=tamano_numeros_ejes
    )

    # Leyenda
    ax2.legend(
        fontsize=tamano_leyenda
    )


    # ========================================================
    # TÍTULO GENERAL DE LA FIGURA
    # ========================================================

    fig.suptitle(
        titulo_general.format(
            zona=zona
        ),
        fontsize=tamano_titulo_general,
        y=1.02
    )


    # ========================================================
    # AJUSTAR ESPACIADO
    # ========================================================

    plt.tight_layout()


    # ========================================================
    # GUARDAR IMAGEN
    # ========================================================

    archivo_png = (
        carpeta_salida /
        f"bland_altman_{zona}.png"
    )

    plt.savefig(
        archivo_png,
        dpi=dpi_salida,
        bbox_inches="tight"
    )

    plt.close()


    # ========================================================
    # MOSTRAR RESULTADO
    # ========================================================

    print(
        f"Guardado: {archivo_png}"
    )


    # ========================================================
    # RESUMEN DE RESULTADOS
    # ========================================================

    print(
        f"\n{zona} - EXTRACCIÓN"
    )

    print(
        f"Media: {media_ext:.2f}"
    )

    print(
        f"LS: {ls_ext:.2f}"
    )

    print(
        f"LI: {li_ext:.2f}"
    )


    print(
        f"\n{zona} - RIPIOS"
    )

    print(
        f"Media: {media_rip:.2f}"
    )

    print(
        f"LS: {ls_rip:.2f}"
    )

    print(
        f"LI: {li_rip:.2f}"
    )


# ============================================================
# FINALIZACIÓN
# ============================================================

print("\n" + "=" * 60)
print("PROCESO FINALIZADO")
print("=" * 60)

print(
    "Se generó una imagen por cada zona mineralógica."
)

print(
    "Cada imagen contiene los dos gráficos Bland-Altman:"
)

print(
    "1. Análisis de extracción"
)

print(
    "2. Análisis de ripios"
)

print(
    "Ambos gráficos de cada zona utilizan el mismo rango Y."
)

print(
    "Los tamaños de letra fueron configurados para facilitar "
    "su lectura en documentos y tesis."
)

print("=" * 60)